In [ ]:
"""
05_ocr_analysis.ipynb
======================
Copy each section between # %% markers into separate notebook cells.
Update paths to your actual directories.
"""

# 05 - OCR Analysis
Compares PaddleOCR vs EasyOCR on detected plate crops.

In [1]:
import os
import cv2
import json
import time
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict
from ultralytics import YOLO

# ─── UPDATE THESE PATHS ───
MODEL_PATH = "../training_logs/exp_001_yolo26s_full_finetune/weights/best.pt"
TEST_IMAGES_DIR = r"C:\Users\rajcr\Desktop\Study\MTech\Courses\CV\Project\Dataset\leaked_data_split\test\images"
OCR_MODELS_DIR = "../../submission/ROLL_NUMBER/models"  # where PaddleOCR/EasyOCR models are
GROUND_TRUTH_PATH = None  # set to path if you have GT plate annotations
# ───────────────────────────

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "axes.grid": True,
    "grid.alpha": 0.3,
})

SAVE_DIR = os.path.join(os.path.dirname(TEST_IMAGES_DIR), "..", "ocr_analysis_plots")
os.makedirs(SAVE_DIR, exist_ok=True)

CLASS_PLATE = 3  # number-plate class ID


def save_fig(fig, name):
    fig.savefig(os.path.join(SAVE_DIR, f"{name}.png"), dpi=150, bbox_inches="tight")
    print(f"Saved: {name}.png")


def levenshtein_distance(s1, s2):
    if len(s1) < len(s2):
        return levenshtein_distance(s2, s1)
    if len(s2) == 0:
        return len(s1)
    prev_row = range(len(s2) + 1)
    for i, c1 in enumerate(s1):
        curr_row = [i + 1]
        for j, c2 in enumerate(s2):
            curr_row.append(min(prev_row[j + 1] + 1, curr_row[j] + 1, prev_row[j] + (c1 != c2)))
        prev_row = curr_row
    return prev_row[-1]


def normalized_edit_score(pred, gt):
    if not pred and not gt:
        return 1.0
    max_len = max(len(pred), len(gt))
    if max_len == 0:
        return 1.0
    return 1.0 - levenshtein_distance(pred, gt) / max_len

In [2]:
print("Detecting plates in test images...")
model = YOLO(MODEL_PATH)

import glob
test_images = sorted(glob.glob(os.path.join(TEST_IMAGES_DIR, "*.jpg")))
print(f"Test images: {len(test_images)}")

plate_crops = []  # list of {image, crop, bbox, conf}

for img_path in test_images:
    img = cv2.imread(img_path)
    if img is None:
        continue

    results = model(img_path, conf=0.25, verbose=False)
    if not results or results[0].boxes is None:
        continue

    boxes = results[0].boxes
    for i in range(len(boxes)):
        if int(boxes.cls[i]) == CLASS_PLATE:
            bbox = boxes.xyxy[i].cpu().numpy().astype(int)
            conf = float(boxes.conf[i])
            h, w = img.shape[:2]

            # Crop with 15% padding
            pad_w = int((bbox[2] - bbox[0]) * 0.15)
            pad_h = int((bbox[3] - bbox[1]) * 0.15)
            x1 = max(0, bbox[0] - pad_w)
            y1 = max(0, bbox[1] - pad_h)
            x2 = min(w, bbox[2] + pad_w)
            y2 = min(h, bbox[3] + pad_h)

            crop = img[y1:y2, x1:x2]
            if crop.size > 0:
                plate_crops.append({
                    "image": os.path.basename(img_path),
                    "crop": crop,
                    "bbox": bbox.tolist(),
                    "det_conf": conf,
                })

print(f"Extracted {len(plate_crops)} plate crops")

Detecting plates in test images...
Test images: 653
Extracted 441 plate crops


In [3]:
def preprocess_plate(crop):
    """Same preprocessing as solution.py"""
    target_height = 64
    h, w = crop.shape[:2]
    if h == 0:
        return None
    scale = target_height / h
    new_w = max(1, int(w * scale))
    resized = cv2.resize(crop, (new_w, target_height), interpolation=cv2.INTER_CUBIC)

    if len(resized.shape) == 3:
        gray = cv2.cvtColor(resized, cv2.COLOR_BGR2GRAY)
    else:
        gray = resized

    clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
    enhanced = clahe.apply(gray)
    enhanced = cv2.GaussianBlur(enhanced, (3, 3), 0)
    return enhanced


preprocessed_crops = []
for pc in plate_crops:
    pp = preprocess_plate(pc["crop"])
    if pp is not None:
        pc["preprocessed"] = pp
        preprocessed_crops.append(pc)

print(f"Preprocessed: {len(preprocessed_crops)} crops")

Preprocessed: 441 crops


In [4]:
from paddleocr import PaddleOCR
import easyocr

paddle_ocr = PaddleOCR(
    use_angle_cls=True,
    lang="en",
    det_model_dir=os.path.join(OCR_MODELS_DIR, "ocr_det") if os.path.isdir(os.path.join(OCR_MODELS_DIR, "ocr_det")) else None,
    rec_model_dir=os.path.join(OCR_MODELS_DIR, "ocr_rec") if os.path.isdir(os.path.join(OCR_MODELS_DIR, "ocr_rec")) else None,
    cls_model_dir=os.path.join(OCR_MODELS_DIR, "ocr_cls") if os.path.isdir(os.path.join(OCR_MODELS_DIR, "ocr_cls")) else None,
    show_log=False,
)

easyocr_dir = os.path.join(OCR_MODELS_DIR, "easyocr_models")
easy_ocr = easyocr.Reader(
    ["en"],
    gpu=False,
    model_storage_directory=easyocr_dir if os.path.isdir(easyocr_dir) else None,
    download_enabled=True if not os.path.isdir(easyocr_dir) else False,
)

print("Both OCR engines initialized")

c:\Users\rajcr\AppData\Local\Programs\Python\Python312\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (7.4.3)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
C:\Users\rajcr\AppData\Local\Temp\ipykernel_1816\1084184131.py:4: DeprecationWarning: The parameter `use_angle_cls` has been deprecated and will be removed in the future. Please use `use_textline_orientation` instead.
  paddle_ocr = PaddleOCR(
C:\Users\rajcr\AppData\Local\Temp\ipykernel_1816\1084184131.py:4: DeprecationWarning: The parameter `det_model_dir` has been deprecated and will be removed in the future. Please use `text_detection_model_dir` instead.
  paddle_ocr = PaddleOCR(
C:\Users\rajcr\AppData\Local\Temp\ipykernel_1816\1084184131.py:4: DeprecationWarning: The parameter `rec_model_dir` has been deprecated and will be removed in the future. Please use `text_recognition_model_dir` instead.
  paddle_ocr = PaddleOCR(
C:\Users\rajcr\AppData\Loc

ValueError: Unknown argument: show_log

In [ ]:
def run_paddle(crop):
    try:
        result = paddle_ocr.ocr(crop, cls=True)
        if result and result[0]:
            texts, confs = [], []
            for line in result[0]:
                if line and len(line) >= 2:
                    texts.append(line[1][0])
                    confs.append(line[1][1])
            if texts:
                return " ".join(texts), sum(confs) / len(confs)
    except Exception:
        pass
    return "", 0.0


def run_easy(crop):
    try:
        result = easy_ocr.readtext(crop)
        if result:
            texts, confs = [], []
            for det in result:
                texts.append(det[1])
                confs.append(det[2])
            if texts:
                return " ".join(texts), sum(confs) / len(confs)
    except Exception:
        pass
    return "", 0.0


print("Running OCR on all plate crops...")
ocr_results = []

for i, pc in enumerate(preprocessed_crops):
    crop = pc["preprocessed"]

    # PaddleOCR
    t0 = time.time()
    p_text, p_conf = run_paddle(crop)
    p_time = time.time() - t0

    # EasyOCR
    t0 = time.time()
    e_text, e_conf = run_easy(crop)
    e_time = time.time() - t0

    # Dual strategy (as in solution.py)
    if p_conf > 0.3:
        dual_text = p_text
        dual_source = "paddle"
    elif e_conf > 0.3:
        dual_text = e_text
        dual_source = "easy"
    elif p_conf >= e_conf:
        dual_text = p_text
        dual_source = "paddle_low"
    else:
        dual_text = e_text
        dual_source = "easy_low"

    result = {
        "image": pc["image"],
        "det_conf": pc["det_conf"],
        "paddle_text": p_text.strip().upper(),
        "paddle_conf": p_conf,
        "paddle_time": p_time,
        "easy_text": e_text.strip().upper(),
        "easy_conf": e_conf,
        "easy_time": e_time,
        "dual_text": dual_text.strip().upper(),
        "dual_source": dual_source,
    }
    ocr_results.append(result)

    if (i + 1) % 20 == 0:
        print(f"  Processed {i+1}/{len(preprocessed_crops)}")

print(f"Done. {len(ocr_results)} plates processed.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

paddle_confs = [r["paddle_conf"] for r in ocr_results]
easy_confs = [r["easy_conf"] for r in ocr_results]

axes[0].hist(paddle_confs, bins=20, color="#1976D2", alpha=0.7, edgecolor="black")
axes[0].axvline(0.3, color="red", linestyle="--", label="Threshold (0.3)")
axes[0].set_title(f"PaddleOCR Confidence (mean={np.mean(paddle_confs):.3f})")
axes[0].set_xlabel("Confidence")
axes[0].set_ylabel("Count")
axes[0].legend()

axes[1].hist(easy_confs, bins=20, color="#4CAF50", alpha=0.7, edgecolor="black")
axes[1].axvline(0.3, color="red", linestyle="--", label="Threshold (0.3)")
axes[1].set_title(f"EasyOCR Confidence (mean={np.mean(easy_confs):.3f})")
axes[1].set_xlabel("Confidence")
axes[1].legend()

fig.suptitle("OCR Confidence Distribution", fontsize=14)
fig.tight_layout()
save_fig(fig, "01_confidence_distribution")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))

paddle_times = [r["paddle_time"] * 1000 for r in ocr_results]  # ms
easy_times = [r["easy_time"] * 1000 for r in ocr_results]

positions = [1, 2]
bp = ax.boxplot([paddle_times, easy_times], positions=positions, widths=0.5,
                patch_artist=True)
bp["boxes"][0].set_facecolor("#1976D2")
bp["boxes"][1].set_facecolor("#4CAF50")

ax.set_xticks(positions)
ax.set_xticklabels(["PaddleOCR", "EasyOCR"])
ax.set_ylabel("Time (ms)")
ax.set_title(f"OCR Inference Time (PaddleOCR: {np.mean(paddle_times):.0f}ms, EasyOCR: {np.mean(easy_times):.0f}ms)")
fig.tight_layout()
save_fig(fig, "02_inference_time")
plt.show()

In [ ]:
from collections import Counter

source_counts = Counter(r["dual_source"] for r in ocr_results)

fig, ax = plt.subplots(figsize=(8, 5))
labels = list(source_counts.keys())
values = list(source_counts.values())
colors_pie = {"paddle": "#1976D2", "easy": "#4CAF50", "paddle_low": "#90CAF9", "easy_low": "#A5D6A7"}
pie_colors = [colors_pie.get(l, "#999999") for l in labels]

ax.pie(values, labels=labels, autopct="%1.1f%%", colors=pie_colors, startangle=90)
ax.set_title("Dual OCR — Which Engine Was Selected")
fig.tight_layout()
save_fig(fig, "03_dual_source_distribution")
plt.show()

In [ ]:
agreements = 0
disagreements = 0
only_paddle = 0
only_easy = 0
both_empty = 0

for r in ocr_results:
    p = r["paddle_text"]
    e = r["easy_text"]

    if not p and not e:
        both_empty += 1
    elif p == e:
        agreements += 1
    elif p and not e:
        only_paddle += 1
    elif e and not p:
        only_easy += 1
    else:
        disagreements += 1

fig, ax = plt.subplots(figsize=(8, 5))
categories = ["Agree", "Disagree", "Only Paddle", "Only Easy", "Both Empty"]
values = [agreements, disagreements, only_paddle, only_easy, both_empty]
bar_colors = ["#4CAF50", "#F44336", "#1976D2", "#66BB6A", "#BDBDBD"]

ax.bar(categories, values, color=bar_colors)
ax.set_ylabel("Count")
ax.set_title("OCR Agreement Analysis")
for i, v in enumerate(values):
    ax.text(i, v + 0.5, str(v), ha="center", fontsize=11)
fig.tight_layout()
save_fig(fig, "04_agreement_analysis")
plt.show()

In [ ]:
if GROUND_TRUTH_PATH and os.path.exists(GROUND_TRUTH_PATH):
    with open(GROUND_TRUTH_PATH) as f:
        gt_plates = json.load(f)

    paddle_exact, easy_exact, dual_exact = 0, 0, 0
    paddle_edit, easy_edit, dual_edit = [], [], []
    matched = 0

    for r in ocr_results:
        gt = gt_plates.get(r["image"], "")
        if not gt:
            continue
        gt = gt.strip().upper()
        matched += 1

        if r["paddle_text"] == gt:
            paddle_exact += 1
        if r["easy_text"] == gt:
            easy_exact += 1
        if r["dual_text"] == gt:
            dual_exact += 1

        paddle_edit.append(normalized_edit_score(r["paddle_text"], gt))
        easy_edit.append(normalized_edit_score(r["easy_text"], gt))
        dual_edit.append(normalized_edit_score(r["dual_text"], gt))

    if matched > 0:
        fig, axes = plt.subplots(1, 2, figsize=(14, 5))

        # Exact match
        ax = axes[0]
        engines = ["PaddleOCR", "EasyOCR", "Dual"]
        exact_vals = [paddle_exact/matched*100, easy_exact/matched*100, dual_exact/matched*100]
        bars = ax.bar(engines, exact_vals, color=["#1976D2", "#4CAF50", "#FF9800"])
        ax.set_ylabel("Exact Match %")
        ax.set_title("Exact Match Rate")
        for bar, v in zip(bars, exact_vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f"{v:.1f}%", ha="center", fontsize=11)

        # Edit distance
        ax = axes[1]
        edit_vals = [np.mean(paddle_edit), np.mean(easy_edit), np.mean(dual_edit)]
        bars = ax.bar(engines, edit_vals, color=["#1976D2", "#4CAF50", "#FF9800"])
        ax.set_ylabel("Avg Normalized Edit Score")
        ax.set_title("Edit Distance Score (higher = better)")
        ax.set_ylim(0, 1)
        for bar, v in zip(bars, edit_vals):
            ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                    f"{v:.3f}", ha="center", fontsize=11)

        fig.suptitle(f"OCR Accuracy Comparison (n={matched} plates)", fontsize=14)
        fig.tight_layout()
        save_fig(fig, "05_accuracy_comparison")
        plt.show()

        print(f"\n  Accuracy Summary ({matched} plates with GT):")
        print(f"  {'Engine':<15} {'Exact Match':>12} {'Edit Score':>12}")
        print(f"  {'PaddleOCR':<15} {paddle_exact/matched*100:>11.1f}% {np.mean(paddle_edit):>11.3f}")
        print(f"  {'EasyOCR':<15} {easy_exact/matched*100:>11.1f}% {np.mean(easy_edit):>11.3f}")
        print(f"  {'Dual (ours)':<15} {dual_exact/matched*100:>11.1f}% {np.mean(dual_edit):>11.3f}")
else:
    print("No ground truth available. Skipping accuracy comparison.")
    print("Set GROUND_TRUTH_PATH to a JSON file mapping image_name → plate_text")

In [ ]:
n_samples = min(20, len(preprocessed_crops))
fig, axes = plt.subplots(4, 5, figsize=(20, 12))

for idx, ax in enumerate(axes.flat):
    if idx >= n_samples:
        ax.axis("off")
        continue

    pc = preprocessed_crops[idx]
    r = ocr_results[idx]

    # Show original crop (not preprocessed)
    crop_rgb = cv2.cvtColor(pc["crop"], cv2.COLOR_BGR2RGB)
    ax.imshow(crop_rgb)

    title = f"P: {r['paddle_text'][:15]} ({r['paddle_conf']:.2f})\n"
    title += f"E: {r['easy_text'][:15]} ({r['easy_conf']:.2f})"
    ax.set_title(title, fontsize=8)
    ax.axis("off")

fig.suptitle("Sample Plate Crops with OCR Results (P=Paddle, E=Easy)", fontsize=14)
fig.tight_layout()
save_fig(fig, "06_sample_plates")
plt.show()

In [ ]:
print("=" * 60)
print("  OCR ANALYSIS SUMMARY")
print("=" * 60)
print(f"\n  Total plates detected: {len(ocr_results)}")
print(f"\n  PaddleOCR:")
print(f"    Mean confidence:  {np.mean(paddle_confs):.3f}")
print(f"    Above 0.3:        {sum(1 for c in paddle_confs if c > 0.3)}/{len(paddle_confs)}")
print(f"    Mean time:        {np.mean(paddle_times):.0f}ms")
print(f"\n  EasyOCR:")
print(f"    Mean confidence:  {np.mean(easy_confs):.3f}")
print(f"    Above 0.3:        {sum(1 for c in easy_confs if c > 0.3)}/{len(easy_confs)}")
print(f"    Mean time:        {np.mean(easy_times):.0f}ms")
print(f"\n  Dual Strategy:")
print(f"    Source: {dict(source_counts)}")
print(f"    Agreement rate:   {agreements}/{len(ocr_results)} ({agreements/max(len(ocr_results),1)*100:.1f}%)")